# Week 13. Natural Language Processing (NLP)

In this lecture, you will be introduced to using Natural Language Processing (NLP) in urban analytics.

## What is NLP and how can you use it?

NLP is ability to process text or spoken word based data with a computer in order to efficiently deal with large, potentially unruly or unstructured, data. 

In urban analytics, the uses of NLP are boundless! You can now handle large amounts of data coming from plans themselves, online open response questionaires, social media postings, transcripts from interviews or meetings, and more. Each of these datasets can illuminate important themes that may be difficult or time consuming to find by hand.

The NLP processing chain is most often:
1. Preprocess data to make text as uniform as possible.
2. Decide what each "document" should be - whole body, paragraph, sentence, few words, etc.
3. Turn each document into vector.
4. Utilize various existing tools with vectorized data.
5. Analyze results!

In [ ]:
!pip install nltk

In [ ]:
import nltk
#nltk.download()

In [ ]:
import re
import string
import numpy as np
from collections import Counter
from itertools import chain
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer


In [ ]:
# Let's work with our labeled listings!
data = pd.read_csv('labeled_data.csv')
data['body'].iloc[0]

## 1. Preprocessing data

Ultimately, you are working with one string composed of different symbols (letters and numbers), so creating uniformity however possible is helpful. You want the computer to recognize "T" and "t" as the same symbol. You might not also want your application to care about "*" or "|". It all depends on what you want to pick up on.

### 1.1 Regex Commands

#### - forcing lowercase and removing unwanted symbols

A great tool for manipulating string/text based data is regex. "regex" is shorthand for "regular expression" and is a programming language in itself. There are forms of composing a regex command in order to change a string into something else. 

Some important regex commands:
1. ```[ ]``` tells it to look for character types that fall in between the brackets
2. ```^``` tells it to perform an action on anything EXCEPT the characters that follow ^
3. ```*``` tells it to look for a character zero or more times. Ex. ```ca*t``` will match ```ct, cat, caat,``` etc.
4. ```+``` is similar to ```*```, but it looks for a match of one or more times. Ex. ```ca+t``` will only match ```cat, caat,``` etc.
5. ```?``` tells it that a character is optional, or that it can be there zero or one times. Ex. ```ca?t``` will match ```ct, cat```
6. ```a-c``` tells it to look for all characters that fall between the two given, with numerical or alphabetical order. Ex. ```a-z``` will look at the full alphabet, ```0-9```will look at all digits.

Using regex: 
1. ```re.compile``` This can be used to save out a regex command. Think of it kind of like a function. Ex. ```t = re.compile('[0-9]+')``` saves the command "look for all digits if they appear at least once" as variable "t"
2. ```re.findall``` Now, we can use our compiled command against sets of text data to find all instances that match the command. Ex. ```t.findall('I have 10 oranges, 8 apples, and 6 pears')``` will return ```['10','8','6']```
3. ```re.sub``` We can also change what the text says with regex. Ex. ```t.sub('number', 'I have 10 oranges, 8 apples, and 6 pears')``` will return ```'I have number oranges, number apples, and number pears'```. You can also call re.sub directly with ```re.sub(expression, replacement, string)```

In [ ]:
#isolate the text column
bodytext = data['body']
#make all letters lowercase
bodytext = bodytext.str.lower()
#remove non alphabetic characters 
bodytext = bodytext.apply(lambda x: re.sub("[^A-Za-z]+", ' ', str(x)))

In [ ]:
#view the before
print(data['body'].iloc[0])

In [ ]:
#view the after
print(bodytext.iloc[0])

In [ ]:
## YOUR TURN
## Write a regex command to sub all the As (upper and lower case) with the symbol & in the following string
exp = "I have two pals named Alice"



### 1.2 Introduction to nltk

One of the most powerful tools in Python for NLP is the natural langauge toolkit (nltk) (https://www.nltk.org/). It is rich with processes and easy to use. Often, this package is used for the preprocessing stage where your text data may undergo any of the following:

#### - removing "stopwords"

Stopwords are very common, usually insignificant words that you want filtered out before you do any processing.

In [ ]:
#Take a look at some of the "stopwords"
nltk.corpus.stopwords.words('english')[0:10]

In [ ]:
#remove these from each document
bodytext = bodytext.apply(lambda x: x.split(" "))
no_stopwords = bodytext.apply(lambda x: sorted(set(x) - set(nltk.corpus.stopwords.words('english')), key=x.index))

In [ ]:
#now view our sample text without any stopwords 
print(no_stopwords.iloc[0])

#### - stemming or lemmatizing 

Stemming is the process of taking a word down to its root. Lemmatizing is the process of changing a word to its base format. Either step is usually performed in order to help your model capture variations in how people might represent words. For example, if you wanted to know how often people were talking about change in a system, you would want to capture whenever people say "change", "changing", "changes", or "changed". You can see how this would happen for stemming vs lemmatizing below.

| Stemming | Lemmatizing |
| --- | --- | 
| change $\rightarrow$ chang | change $\rightarrow$ change | 
| changes $\rightarrow$ chang | changes $\rightarrow$ change | 
| changing $\rightarrow$ chang | changing $\rightarrow$ change | 
| changed $\rightarrow$ chang | changed $\rightarrow$ change | 

In [ ]:
#stem each word 
#initialze Stemmer
stemmer = nltk.stem.PorterStemmer()
#apply to each word in each document
bodytext_stemmed = bodytext.apply(lambda x: [stemmer.stem(i) for i in x])

In [ ]:
#view our sample text after being stemmed
print(bodytext_stemmed.iloc[0])

In [ ]:
#lemmatize each word
#initialize Lemmatizer
wnl = nltk.stem.WordNetLemmatizer()
#apply to each word in each document
bodytext_lemm = bodytext.apply(lambda x: [wnl.lemmatize(i) for i in x])

In [ ]:
#view our sample text after being lemmatized
print(bodytext_lemm.iloc[0])

#### NLTK has other powerful accessories!

nltk can help identify the part of speech to isolate nouns, verbs, adjectives, etc. It can also identify groupings of words that most often occur together!

The nltk POS codes are: 


| Code | Part of Speech | Code | Part of Speech |
| --- | --- | --- | --- |  
|CC:| conjunction, coordinating |PDT:| pre-determiner |
|CD:| numeral, cardinal |POS:| genitive marker |
|DT:| determiner |PRP:| pronoun, personal |
|EX:| existential there |RB:| adverb |
|IN:| preposition or conjunction, subordinating |RP:| particle |
|JJ:| adjective or numeral, ordinal |TO:| "to" as preposition or infinitive marker |
|JJR:| adjective, comparative |UH:| interjection |
|JJS:| adjective, superlative |VB:| verb, base form |
|LS:| list item marker |VBD:| verb, past tense |
|MD:| modal auxiliary |VBG:| verb, present participle or gerund |
|NN:| noun, common, singular or mass |VBN:| verb, past participle |
|NNP:| noun, proper, singular | WDT:| WH-determiner |
|NNS:| noun, common, plural|

In [ ]:
# Identify the part of speech and isolate adjectives, nouns, etc.
example_sentence = bodytext.iloc[0]
print(nltk.pos_tag(example_sentence))

In [ ]:
#look at all of the adjectives for the postings
def keep_pos(x,pos=['JJ','JJS','JJR']):
    """
    Keeps only the parts of speech identified in pos.
    Returns the words that fit the criteria.
    """
    tagged = nltk.pos_tag(x)
    words_to_keep = [t[0] for t in tagged if t[1] in pos]
    return words_to_keep

keep_pos(example_sentence, pos=['JJ','JJS','JJR'])

In [ ]:
# Identify words that often appear together
number_of_words = 10
ngrams = no_stopwords.apply(lambda x: list(nltk.ngrams(x,number_of_words)))
count = Counter(list(chain.from_iterable(list(ngrams.values))))

In [ ]:
count.most_common(5)

In [ ]:
## YOUR TURN
## Identify all of the nouns in our example sentence


## 2. Determining Your "Document"

Step 2 of the NLP process is determining what your "document" will be. This can be the whole text as one, each sentence individually, or even bi- or tri-grams of words. 

In [ ]:
#split by sentence
def split_by_sent(text, split_criteria=['  ','.', '!', '?','\n']):
    """
    Take in text, and some list of criteria to 
    determine when a new sentence starts.
    Output a list of the sentences.
    """
    for x in split_criteria:
        text = str(text).replace(x, '*')
    bodylist = str(text).split('*')
    bodylist = [w for w in bodylist if w != '']
    return bodylist    
    
sentences = data['body'].str.lower().apply(lambda x: split_by_sent(x))
sentencedf = sentences.explode()
sentencedf = sentencedf[~sentencedf.isna()]
print(sentences.iloc[0])
print('\n', sentencedf.iloc[0])

In [ ]:
#split by bigram
bigrams = no_stopwords.apply(lambda x: list(nltk.ngrams(x,2)))
bigramdf = bigrams.explode()
print(bigrams.iloc[0])
print('\n', bigramdf.iloc[0])

## 3. TF-IDF Vectors
One method of performing step 3, turning each document into a vector, is through Term Frequency-Inverse Document Frequency (TFIDF). TF-IDF measures how important each word is to each document. 

Term Frequency (tf) refers to how often a word occurs in a document, ranging from 0 to 1. Inverse document frequency (idf) refers to how often a word occurs in _any_ of the documents, where closer to 0 represents more common words (think: and, the, it) and closer to 1 represents rarer words (think: quire, ulotrichous).

The goal is to have a vector for each document that is 1 x n (n being the total number of words in the dataset dictionary) with values describing the tf * idf scores for each word.

In [ ]:
#First, we need a vector that shows the counts of each word in each document. Most of it will be 0.
documents = bodytext.apply(lambda x: ' '.join(x))
count_vect = CountVectorizer()
data_counts = count_vect.fit_transform(documents)

tf_feature_names = count_vect.get_feature_names_out()
tf_feature_names[0:200]

In [ ]:
#Then, we can create the tf-idf matrix
tfidf_transformer = TfidfTransformer()
data_tfidf = tfidf_transformer.fit_transform(data_counts)

tfidf_feature_names = tfidf_transformer.get_feature_names_out()
tfidf_feature_names

In [ ]:
#Inspect the shape of the matrix
print(data_counts.shape)
print(data_tfidf.shape)

In [ ]:
#Now with the sentence dataframe
#First, we need a vector that shows the counts of each word in each document. Most of it will be 0.
count_vect_sent = CountVectorizer()
data_counts_sentences = count_vect_sent.fit_transform(sentencedf)
#Then, we can create the tf-idf matrix
tfidf_transformer_sent = TfidfTransformer()
data_tfidf_sentences = tfidf_transformer_sent.fit_transform(data_counts_sentences)
#Inspect the shape of the matrix
print(data_counts_sentences.shape)
print(data_tfidf_sentences.shape)

## 4. Topic Modeling

Now that we have our documents represented as a matrix (m documents x n words in dictionary), we want to understand what topics are present 

### 4.1 Latent Dirichlet Allocation (LDA)

LDA is an unsupervised topic modeling technique. We can use this technique to create clusters, or topics, that are commonly occuring across all of the documents. Then, we can understand what words describe those topics. Finally, we can trace the topics back to our documents (remember, this can be the full ad or a single sentence) and see what topics appear in each document. There can be more than one topic per document!   

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation

#preprocess our listings

#lowercase
bodytext = data['body'].str.lower()

#remove new lines and tabs, numbers, and stopwords
bodytext = bodytext.apply(lambda x: str(x).replace("\n", "").replace("\t", "").replace("\xa0", ""))
bodytext = bodytext.apply(lambda x: str(x).split(" "))
bodytext = bodytext.apply(lambda x: sorted(set(x) - set(nltk.corpus.stopwords.words('english')), key=x.index))

#stem words
bodytext = bodytext.apply(lambda x: [stemmer.stem(i) for i in x])

#split into sentences
sentences = bodytext.apply(lambda x: split_by_sent(" ".join(x)))
documents = sentences.explode()

# remove empty sentences 
documents = documents[~documents.isna()]
documents = documents.apply(lambda x: x.split(" "))
documents = documents.apply(lambda x: [w for w in x if len(w)>0])
documents = documents[documents.apply(lambda x: len(x)>0)]

# go from list back into string, remove non-alphabetic characters
documents = documents.apply(lambda x: " ".join(x))
documents = documents.apply(lambda x: re.sub("[^A-Za-z']+", ' ', str(x)))
documents.head()

In [ ]:
# Get the count vector
tf_vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
tf = tf_vectorizer.fit_transform(documents)
tf_feature_names = tf_vectorizer.get_feature_names_out()

In [ ]:
#choose number of topics and create model
num_topics = 10
lda = LatentDirichletAllocation(n_components=num_topics, 
                                max_iter = 5,
                                random_state=0)
topic_proportions = lda.fit_transform(tf)

In [ ]:
def display_topics(model, feature_names, no_top_words):
    """
    Read in the model and feature names and return the top
    N words determined for each topic.
    No output, prints to console.
    """
    for topic_idx, topic in enumerate(model.components_):
        print(f"Topic {topic_idx}")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

num_top_words = 10
display_topics(lda, tf_feature_names, num_top_words)

In [ ]:
topic_proportions.shape

In [ ]:
print(documents.iloc[0])
print(topic_proportions[0])

In [ ]:
## YOUR TURN
## Fit an LDA model with more topics (15? 20?) and display the top words per topic



In [ ]:
#save the most probable topic per document

#create a dataframe
top_topics = pd.DataFrame({'adindex':documents.index, # remember the index is set to the original dataframe index
                           'document': documents.values, 
                           'top_topic': topic_proportions.argmax(axis=1), #axis=1 gives us index of max value per row
                           'sent_len':documents.apply(lambda x: len(x.split(" "))) # get number of words in sentence
                          })

top_topics.head()

In [ ]:
#calculate what percentage of the ad is dedicated to each topic 

percentages = np.zeros((len(data),num_topics))
#groupby the ad and the topic of the sentence. Sum the number of words per ad per topic
groupeddf = top_topics.groupby(['adindex', 'top_topic'])['sent_len'].sum()
#Put into a matrix
for idx in groupeddf.index:
    percentages[idx] = groupeddf[idx]
percentages = np.transpose(np.transpose(percentages)/percentages.sum(axis=1))

In [ ]:
#plot the percentage of the ads dedicated to each topic 
pd.DataFrame(data=percentages, columns = range(num_topics)).boxplot()
plt.show()

## 5. Supervised Learning with Text

We can run our supervised models on the text as well. We just need our strings to be represented numerically (TF-IDF)!


For example, let's use our labels and K-NN. 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report


# First let's work with the full listings because that's how we labeled them

#lowercase
data['bodytext'] = data['body'].str.lower()

#remove new lines and tabs, numbers, and stopwords
data['bodytext'] = data['bodytext'].apply(lambda x: str(x).replace("\n", "").replace("\t", "").replace("\xa0", ""))
data['bodytext'] = data['bodytext'].apply(lambda x: str(x).split(" "))
data['bodytext'] = data['bodytext'].apply(lambda x: sorted(set(x) - set(nltk.corpus.stopwords.words('english')), key=x.index))

#stem words
data['bodytext'] = data['bodytext'].apply(lambda x: [stemmer.stem(i) for i in x])

# go from list back into string, remove non-alphabetic characters
data['bodytext'] = data['bodytext'].apply(lambda x: " ".join(x))
data['bodytext'] = data['bodytext'].apply(lambda x: re.sub("[^A-Za-z']+", ' ', str(x)))
data['bodytext'].head()


In [ ]:
# And let's examine our labels
data['LABEL'].value_counts()

In [ ]:
# We have messy labels!
data['LABEL'].unique()

In [ ]:
# Fix these!

# First remove spaces
data['LABEL_clean'] = data['LABEL'].apply(lambda x: str(x).replace(" ",""))
# Change A to ADU ? 
data['LABEL_clean'] = data['LABEL_clean'].apply(lambda x: "ADU" if x == 'A' else x)

# Remove S ADU because we it doesn't fit nicely into one of our categories
data['LABEL_clean'] = data['LABEL_clean'].apply(lambda x: np.nan if x == 'SADU' or x == '' else x)

# drop these NaNs
data = data.dropna(subset='LABEL_clean')

data['LABEL_clean'].value_counts()

In [ ]:
# We will split into training and testing

# Get each training and testing set 
df_train, df_test = train_test_split(data, train_size=0.75, stratify=data["LABEL_clean"])

df_train.head()

In [ ]:
# Build our TF-IDF pipeline
count_vect = CountVectorizer()
tfidf_transformer = TfidfTransformer()

# And recall our KNN model
knn = KNeighborsClassifier(n_neighbors=5)

# Now we can build them into one pipeline so data always gets preprocessed the same way 
knn_pipeline = make_pipeline(count_vect, tfidf_transformer, knn)

knn_pipeline.fit(X=df_train["bodytext"], y=df_train["LABEL_clean"])


predicted = knn_pipeline.predict(df_test["bodytext"])
df_test['predicted'] = predicted # Add new column
df_test[["LABEL_clean", "predicted"]].head(10) # Show side by side

In [ ]:
# How well does our model perform?
print(classification_report(df_test["LABEL_clean"], df_test["predicted"]))

In [ ]:
# Which labels are getting confused with each other?
pd.crosstab(df_test['LABEL_clean'], df_test['predicted'])

#### Running our model on all of our data!

In [ ]:
# Let's build our preprocessing steps into a function so we can use it in our pipeline
from sklearn.preprocessing import FunctionTransformer

def clean_text(X):
    """
    Take in the body of a listing and clean it. 
    Outputs the new, cleaned, body text.
    """
    X = X.str.lower()
    
    #remove new lines and tabs, numbers, and stopwords
    X = X.apply(lambda x: str(x).replace("\n", "").replace("\t", "").replace("\xa0", ""))
    X = X.apply(lambda x: str(x).split(" "))
    X = X.apply(lambda x: sorted(set(x) - set(nltk.corpus.stopwords.words('english')), key=x.index))
    
    # go from list back into string, remove non-alphabetic characters
    X = X.apply(lambda x: " ".join(x))
    X = X.apply(lambda x: re.sub("[^A-Za-z']+", ' ', str(x)))
    return X


# And we have our other pipeline steps
# Build our TF-IDF pipeline
count_vect = CountVectorizer()
tfidf_transformer = TfidfTransformer()

# And recall our KNN model
knn = KNeighborsClassifier(n_neighbors=5)

# Now we can build them into one pipeline so data always gets preprocessed the same way 
knn_pipeline = make_pipeline(FunctionTransformer(clean_text),
                             count_vect,
                             tfidf_transformer, 
                             knn)


knn_pipeline.fit(X=df_train["body"], y=df_train["LABEL_clean"])


In [ ]:
# Read in our full data 
fulldata = pd.read_csv('la_master.csv')
fulldata.head()

In [ ]:
# Run the body through our pipeline with "predict"
fulldata['predicted'] = knn_pipeline.predict(fulldata["body"])

In [ ]:
# How many listings are predicted in each category?
fulldata['predicted'].value_counts()

In [ ]:
# By percentage
fulldata['predicted'].value_counts(normalize=True)

In [ ]:
# Qualitatively, how do we do?
print(fulldata[fulldata['predicted'] == 'M'].body.iloc[100])

In [ ]:
## YOUR TURN
## Look at the other columns. What can you learn about multi-family units vs single-family units? 
## Produce a plot or a table looking at differences between these two categories

